In [68]:
import pandas as pd
import numpy as np

In [69]:

event_logs = pd.read_csv('../raw_datasets/event_logs.csv')
marketing_summary = pd.read_csv('../raw_datasets/marketing_summary.csv')
trend_report = pd.read_csv('../raw_datasets/trend_report.csv')

# 1. Event Logs

In [70]:
#Remove mess data
event_logs = event_logs.loc[:, :'amount']
event_logs.head()

,user_id,event_type,event_time,product_id,amount
0,U0099,checkout,2023-06-03 04:13,P010,NaN
1,U0240,wishlist_add,2023-06-03 05:08,P020,2900.63
2,U0374,profile_update,2023-06-05 06:22,P028,NaN
3,U0122,page_view,2023-06-06 03:45,P001,NaN
4,U0211,wishlist_add,2023-06-03 12:38,P015,1728.27


In [71]:
event_logs['product_id'].astype(str).unique()

array(['P010', 'P020', 'P028', 'P001', 'P015', 'P018', 'P006', 'P005',
       'P025', 'P026', 'P029', 'P012', 'P027', 'P007', 'P008', 'P004',
       'P024', 'P014', 'P003', 'P019', 'P011', 'P030', 'P017', 'P013',
       'P022', 'P021', 'P002', 'P023', 'P009', 'P016'], dtype=object)

###  User ID  column

In [72]:
# Normalize 
event_logs['user_id'] = event_logs['user_id'].astype(str).str.strip()

#Validate Structure
event_logs['user_id_valid'] = event_logs['user_id'].notna() & (event_logs['user_id'] != 'nan')

# Flag records with missing IDs
event_logs.loc[~event_logs['user_id_valid'], 'user_id'] = 'unknown_user'

### Event_Time

In [73]:
#normalize
event_logs['event_time'] = pd.to_datetime( event_logs['event_time'], errors='coerce' ) 
#avoid future dates/ invalids 
event_logs.loc[ event_logs['event_time'] > pd.Timestamp.now(), 'event_time' ] = pd.NaT

### Event_Type


In [74]:
#Normalize
event_logs['event_type'] = (
    event_logs['event_type']
    .astype(str)
    .str.strip()
    .str.lower()
)
#Define all taxonomy
valid_events = {
    'checkout',
    'wishlist_add',
    'profile_update',
    'page_view',
    'login',
    'logout',
    'search',
    'add_to_cart'
}
#Validate the event types
event_logs['event_type_valid'] = event_logs['event_type'].isin(valid_events)

#handle unknowns
event_logs.loc[~event_logs['event_type_valid'], 'event_type'] = 'unknown_event'

### Product ID

In [75]:
import re

# Normalize
event_logs['product_id'] = (
    event_logs['product_id']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Validate format
pattern = r'^P\d{3}$'

event_logs['product_id_valid'] = event_logs['product_id'].str.match(pattern)

# Handle invalids
event_logs.loc[~event_logs['product_id_valid'], 'product_id'] = None

### Amount

In [76]:
#normalize dtype 

event_logs['amount'] = pd.to_numeric(event_logs['amount'], errors='coerce')

#turn every row with event type that is not checkout or add to cart as 0
event_logs.loc[
    ~event_logs['event_type'].isin(['checkout', 'add_to_cart']),
    'amount'
] = 0

#detect bad checkout data
event_logs['invalid_checkout_amount'] = (
    (event_logs['event_type'] == 'checkout') &
    (event_logs['amount'].isna() | (event_logs['amount'] == 0))
)

# 2. Marketing Summary 

In [77]:
#Remove mess data

marketing_summary = marketing_summary.loc[:, :'report_generated']
marketing_summary.head()


,date,users_active,total_sales,new_customers,report_generated
0,2023-06-01,179,81287.31,9,2023-06-01 16:00
1,2023-06-02,67,74771.99,5,2023-06-02 16:00
2,2023-06-03,369,84809.74,11,2023-06-03 16:00
3,2023-06-04,94,61212.30,3,2023-06-04 16:00
4,2023-06-05,402,80911.49,10,2023-06-05 16:00


### Date

In [78]:
#normalize dtype
marketing_summary['date'] = pd.to_datetime(marketing_summary['date'], errors='coerce')


### users active, new customers and total sales

In [79]:
cols = ['users_active', 'new_customers', 'total_sales']

for col in cols:
    marketing_summary[col] = pd.to_numeric(marketing_summary[col], errors='coerce')
    marketing_summary.loc[marketing_summary[col] < 0, col] = np.nan

### Report Generated

In [80]:
#normalize
marketing_summary['report_generated'] = pd.to_datetime(
    marketing_summary['report_generated'],
    errors='coerce'
)
#avoid future dates/ invalids
marketing_summary.loc[
    marketing_summary['report_generated'] > pd.Timestamp.now(),
    'report_generated'
] = pd.NaT

# 3.Trend Report

In [81]:
#Remove mess data
trend_report = trend_report.loc[:, :'sales_growth_rate']
trend_report.head()

,week,avg_users,sales_growth_rate
0,2023-W21,328,-0.003
1,2023-W22,280,0.088
2,2023-W23,130,0.073
3,2023-W24,291,0.077
4,2023-W25,200,-0.003


### week

In [82]:
# normalize
trend_report['week'] = trend_report['week'].astype(str).str.strip()

# convert to datetime (week start)
trend_report['week_start'] = pd.to_datetime(
    trend_report['week'] + '-1',
    format='%Y-W%U-%w',
    errors='coerce'
)

In [83]:
trend_report.head()

,week,avg_users,sales_growth_rate,week_start
0,2023-W21,328,-0.003,2023-05-22
1,2023-W22,280,0.088,2023-05-29
2,2023-W23,130,0.073,2023-06-05
3,2023-W24,291,0.077,2023-06-12
4,2023-W25,200,-0.003,2023-06-19


### Avg users

In [84]:
trend_report['avg_users'] = pd.to_numeric(trend_report['avg_users'], errors='coerce')
trend_report.loc[trend_report['avg_users'] < 0, 'avg_users'] = np.nan

### Sales Growth Rate'

In [85]:
trend_report['sales_growth_rate'] = pd.to_numeric(trend_report['sales_growth_rate'], errors='coerce')

In [87]:

#export verything
event_logs = event_logs.loc[:, :'amount']

event_logs.to_csv('event_logs(1).csv', index=False)
marketing_summary.to_csv('marketing_summary(1).csv', index=False)
trend_report.to_csv('trend_report(1).csv', index=False)